In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,Path,Label
0,train_0,emoji_dataset/train/train_0.png,2
1,train_1,emoji_dataset/train/train_1.png,3
2,train_2,emoji_dataset/train/train_2.png,4
3,train_3,emoji_dataset/train/train_3.png,3
4,train_4,emoji_dataset/train/train_4.png,3


In [27]:
import torch

epochs = 11
batch_size = 32
learning_rate = 0.0007

device = torch.device("cuda")
device

device(type='cuda')

In [28]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor()
])

valid_transforms = T.Compose([
    T.ToTensor()
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.Path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.df.Label.iloc[idx], dtype = torch.int64).view(1)
        return img, label

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

valid_dataset = MyDataset(df_2, valid_transforms)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(1.)

In [29]:
import torch.nn as nn
import torchvision.models

model = torchvision.models.resnet18(weights=None)
model.fc = nn.LazyLinear(1)

model = model.to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [30]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}")
    print(f"Training Loss: {total_loss / len(train_loader)}")

def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader)}')

In [31]:
for epoch in range(1, epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, valid_loader, criterion)

Epoch 1
Training Loss: 0.8551969245860451
Test Loss: 2.2724795818328856
Epoch 2
Training Loss: 0.4383774007621564
Test Loss: 2.311869978904724
Epoch 3
Training Loss: 0.4015993805308091
Test Loss: 0.5205881893634796
Epoch 4
Training Loss: 0.3270788545671262
Test Loss: 7.339781808853149
Epoch 5
Training Loss: 0.30048104414814397
Test Loss: 0.5666246235370636
Epoch 6
Training Loss: 0.2926660874172261
Test Loss: 0.41629878580570223
Epoch 7
Training Loss: 0.23815561458468437
Test Loss: 0.49021707773208617
Epoch 8
Training Loss: 0.244314462926827
Test Loss: 0.2205355390906334
Epoch 9
Training Loss: 0.2481591376034837
Test Loss: 0.4228960543870926
Epoch 10
Training Loss: 0.2235303201565617
Test Loss: 0.3831268548965454
Epoch 11
Training Loss: 0.21862005560021652
Test Loss: 0.20885798782110215


In [32]:
final_transforms = T.Compose([
    T.ToTensor()
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.Path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transforms)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(1.)

In [33]:
import numpy as np

model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze(1).cpu().numpy()
        outputs = np.round(outputs)
        outputs = np.clip(outputs, 1, 5)
        preds.extend(outputs)

preds[:10]

[3.0, 5.0, 1.0, 4.0, 2.0, 2.0, 3.0, 1.0, 1.0, 3.0]

In [34]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "PredictedLabel": preds
})

df_preds

,SampleID,PredictedLabel
0,test_1500,3.0
1,test_1501,5.0
2,test_1502,1.0
3,test_1503,4.0
4,test_1504,2.0
...,...,...
495,test_1995,3.0
496,test_1996,5.0
497,test_1997,2.0
498,test_1998,2.0


In [35]:
df_preds.to_csv("submission.csv", index=False)